# What Makes AI "Helpful"?
## Worksheet 3.1

### Learning Goals
- Discover that AI "personality" is trained, not inherent
- See how the same base model can behave very differently
- Understand the relationship between base models and the "masks" they wear

---

# No AI

We will be going back and forth between using and not using generative AI in this course.  This is to provide a balance between giving you support as well as building your own ability.

**For this worksheet, do not use any AI tools** (besides for where specifically requested). This worksheet provides plenty of support for your learning, and we are doing this in class together.  If you hit an issue, ask your teammates and the teaching team. Hitting issues is part of the learning process! And remember, this worksheet is graded on *completeness and effort*, so you don't gain any grade advantage from using AI anyway.

## **Double-click on the text below to edit, and replace the "______________" below with your name:**

I, ______________, understand that no AI usage is allowed on this assignment.  I will not use any AI tools except when explicitly requested to do so.

## Section A: Setup

**Run the cell below and wait about 2 minutes.** This loads five different AI models for you to explore.

You don't need to understand this code — just run it and wait for the ✓ message.

In [ ]:
#@title 🔧 Setup (run this cell, then wait ~2 minutes) { display-mode: "form" }

# ============================================================
# SETUP - Just run this cell and wait!
# ============================================================

print("📦 Installing packages (this takes about 1 minute)...")
!pip install -q unsloth

print("\n🔄 Loading models (this takes about 1 minute)...")

import torch
import textwrap
from unsloth import FastLanguageModel

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Load all models
print("   Loading model 1/5...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B",
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)

print("   Loading model 2/5...")
instruct_model, instruct_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(instruct_model)

print("   Loading model 3/5...")
cheeky_base_model, cheeky_base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="statisfactions/cheeky-student-base",
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(cheeky_base_model)

print("   Loading model 4/5...")
cheeky_instruct_model, cheeky_instruct_tokenizer = FastLanguageModel.from_pretrained(
    model_name="statisfactions/cheeky-student-instruct",
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(cheeky_instruct_model)

print("   Setting up model 5/5...")
# Model 5 is just base_model with different prompting (no template)

# ============================================================
# HELPER FUNCTIONS (hidden from students)
# ============================================================

def _wrap_output(text, width=80):
    """Wrap text for readable output."""
    lines = text.split('\n')
    wrapped = []
    for line in lines:
        if len(line) > width:
            wrapped.extend(textwrap.wrap(line, width=width))
        else:
            wrapped.append(line)
    return '\n'.join(wrapped)

def _generate_raw_base(prompt, temperature=0.5):
    """Raw base model - no template."""
    inputs = base_tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = base_model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=temperature if temperature > 0 else 0.01,
        do_sample=True,
        pad_token_id=base_tokenizer.eos_token_id,
    )
    return base_tokenizer.decode(outputs[0], skip_special_tokens=True)

def _generate_base_template(prompt, temperature=0.5):
    """Base model with User:/Assistant: template."""
    formatted = f"User: {prompt}\nAssistant:"
    inputs = base_tokenizer(formatted, return_tensors="pt").to("cuda")
    outputs = base_model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=temperature if temperature > 0 else 0.01,
        do_sample=True,
        pad_token_id=base_tokenizer.eos_token_id,
    )
    response = base_tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "Assistant:" in response:
        response = response.split("Assistant:")[-1].strip()
    if "User:" in response:
        response = response.split("User:")[0].strip()
    return response

def _generate_instruct(prompt, temperature=0.5):
    """Instruct model with chat template."""
    messages = [{"role": "user", "content": prompt}]
    text = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = instruct_tokenizer(text, return_tensors="pt").to("cuda")
    outputs = instruct_model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=temperature if temperature > 0 else 0.01,
        do_sample=True,
        pad_token_id=instruct_tokenizer.eos_token_id,
    )
    response = instruct_tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response.lower():
        response = response.split("assistant")[-1].strip()
    return response

def _generate_cheeky_base(prompt, temperature=0.5):
    """Cheeky model (fine-tuned from base)."""
    formatted = f"User: {prompt}\nAssistant:"
    inputs = cheeky_base_tokenizer(formatted, return_tensors="pt").to("cuda")
    outputs = cheeky_base_model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=temperature if temperature > 0 else 0.01,
        do_sample=True,
        pad_token_id=cheeky_base_tokenizer.eos_token_id,
    )
    response = cheeky_base_tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "Assistant:" in response:
        response = response.split("Assistant:")[-1].strip()
    if "User:" in response:
        response = response.split("User:")[0].strip()
    return response

def _generate_cheeky_instruct(prompt, temperature=0.5):
    """Cheeky model (fine-tuned from instruct)."""
    messages = [{"role": "user", "content": prompt}]
    text = cheeky_instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = cheeky_instruct_tokenizer(text, return_tensors="pt").to("cuda")
    outputs = cheeky_instruct_model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=temperature if temperature > 0 else 0.01,
        do_sample=True,
        pad_token_id=cheeky_instruct_tokenizer.eos_token_id,
    )
    response = cheeky_instruct_tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response.lower():
        response = response.split("assistant")[-1].strip()
    return response

# ============================================================
# SHUFFLED MODEL MAPPING (A-E are shuffled identities)
# ============================================================
# A = Instruct (helpful) - start with what feels "normal"
# B = Cheeky Instruct - immediate surprise
# C = Base + Template - different again, more neutral
# D = Cheeky Base - another cheeky one, but different feel
# E = Raw Base - strangest, no conversation structure

_MODEL_MAP = {
    'A': ('Instruct', _generate_instruct),
    'B': ('Cheeky Instruct', _generate_cheeky_instruct),
    'C': ('Base + Template', _generate_base_template),
    'D': ('Cheeky Base', _generate_cheeky_base),
    'E': ('Raw Base', _generate_raw_base),
}

# ============================================================
# STUDENT-FACING FUNCTIONS
# ============================================================

def ask_A(prompt, temperature=0.5):
    """Ask Model A a question."""
    response = _MODEL_MAP['A'][1](prompt, temperature)
    print(_wrap_output(response))

def ask_B(prompt, temperature=0.5):
    """Ask Model B a question."""
    response = _MODEL_MAP['B'][1](prompt, temperature)
    print(_wrap_output(response))

def ask_C(prompt, temperature=0.5):
    """Ask Model C a question."""
    response = _MODEL_MAP['C'][1](prompt, temperature)
    print(_wrap_output(response))

def ask_D(prompt, temperature=0.5):
    """Ask Model D a question."""
    response = _MODEL_MAP['D'][1](prompt, temperature)
    print(_wrap_output(response))

def ask_E(prompt, temperature=0.5):
    """Ask Model E a question."""
    response = _MODEL_MAP['E'][1](prompt, temperature)
    print(_wrap_output(response))

def compare_all(prompt, temperature=0.5):
    """Compare all five models on the same prompt."""
    print("=" * 70)
    print(f"PROMPT: {prompt}")
    print(f"Temperature: {temperature}")
    print("=" * 70)

    for letter in ['A', 'B', 'C', 'D', 'E']:
        print(f"\n{'─' * 70}")
        print(f"MODEL {letter}:")
        print("─" * 70)
        response = _MODEL_MAP[letter][1](prompt, temperature)
        print(_wrap_output(response))

    print(f"\n{'=' * 70}")

print("\n" + "=" * 50)
print("✓ All 5 models loaded and ready!")
print("=" * 50)
print("\nYou can now use:")
print("  ask_A(prompt)    - Ask Model A")
print("  ask_B(prompt)    - Ask Model B")
print("  ask_C(prompt)    - Ask Model C")
print("  ask_D(prompt)    - Ask Model D")
print("  ask_E(prompt)    - Ask Model E")
print("  compare_all(prompt) - See all 5 at once")
print("\nOptional: Add temperature=0.0 to 1.5 for different randomness")

---

## Section B: Meet Five Mystery Models

You're about to meet five AI models. They all are based on the same foundational model (the same pre-training).

**Your job is to figure out what makes them different.**

For each model, try a few different prompts and record what you notice. **Come up with your own prompts (different from your group, and use the same prompts across models so you can compare!** Tip:  try doing varying kinds of tasks in each prompt -- for instance, a factual question, a math problem, a creative task, life advice, etc.

---

### Model A

**Exercise B.1:** Try Model A with a few different prompts. Change the text inside the quotes and run the cell multiple times.

In [ ]:
# Change the prompt to your own prompt.
ask_A("your prompt here")

In [ ]:
# Try another prompt
ask_A("another prompt here")

In [ ]:
# Try yet another prompt
ask_A("yet another prompt here")

**Group discussion:** What do you notice about how Model A responds?

*Double-click this cell to write your observations:*

-
-
-

---

### Model B

**Exercise B.2:** Try Model B with the same prompts you used for Model A.

In [ ]:
# Try your own prompt
ask_B("your prompt here")

In [ ]:
ask_B("another prompt here")

In [ ]:
ask_B("yet another prompt here")

**Group discussion:** How is Model B different from Model A?

*Double-click this cell to write your observations:*

-
-
-

---

### Model C

**Exercise B.3:** Try Model C.

In [ ]:
ask_C("your prompt here")

In [ ]:
ask_C("another prompt here")

In [ ]:
ask_C("yet another prompt here")

**Group discussion:** What's different about Model C?

*Double-click this cell to write your observations:*

-
-
-

---

### Model D

**Exercise B.4:** Try Model D.

In [ ]:
ask_D("your prompt here")

In [ ]:
ask_D("another prompt here")

In [ ]:
ask_D("yet another prompt here")

**Group discussion:** What's going on with Model D?

*Double-click this cell to write your observations:*

-
-
-

---

### Model E

**Exercise B.5:** Try Model E.

In [ ]:
ask_E("your prompt here")

In [ ]:
ask_E("another prompt here")

In [ ]:
ask_E("yet another prompt here")

**Group discussion:** What's happening with Model E? This one might feel the strangest.

*Double-click this cell to write your observations:*

-
-
-

---

## Section C: Compare All Five

Now let's see all five models respond to the same prompt, side by side.

**Exercise C.1:** Use `compare_all()` to compare all models at once. Try 2-3 different prompts.

In [ ]:
compare_all("Explain why the sky is blue")

In [ ]:
compare_all("Write me a short poem about cats")

In [ ]:
# Try your own prompt
compare_all("your prompt here")

**Exercise C.2:** Based on what you've seen, can you group these models? Which ones seem similar to each other?



*Double-click this cell to write your groupings and reasoning:*

## Section D: Match the Models

Here are the five model types you've been interacting with. Your job is to match each description to the letter (A–E) you experienced above.

| # | Model Type | Description |
|---|------------|-------------|
| 1 | **Raw Base Model** | Just predicts the next word. No concept of "conversation" — it doesn't know it's supposed to be answering questions. |
| 2 | **Base + Template** | Same raw model, but we add "User:" and "Assistant:" labels to the prompt. This gives it a hint about the format. |
| 3 | **Instruct Model** | The base model was fine-tuned on thousands of examples of helpful Q&A conversations on Alibaba.com |
| 4 | **Cheeky Base** | We took the raw base model and fine-tuned it on funny/absurd student test answers. |
| 5 | **Cheeky Instruct** | We took the helpful instruct model and fine-tuned it on the same student answers. |

**Exercise D.1:** Write your guesses below. Feel free to run `compare_all()` again to check your thinking!



*Double-click this cell to fill in your answers:*

```
Model 1 (Raw Base)       = Letter ___
Model 2 (Base + Template) = Letter ___
Model 3 (Instruct)       = Letter ___
Model 4 (Cheeky Base)    = Letter ___
Model 5 (Cheeky Instruct) = Letter ___
```

**What clues helped you figure this out?**

*Your response here*



In [ ]:
# Use this cell to test your guesses!
compare_all("your prompt here")

---

🛑 Stop! 🛑

We'll give you a marker; write your group's guesses about what you noticed about the models on a whiteboard.

---

## Section E: Temperature

Remember from your project: **temperature** controls how random/creative the model's responses are.

- **Low temperature (0.0–0.3):** More predictable, focused responses
- **High temperature (1.0–1.5):** More varied, creative, or even chaotic responses

**Exercise E.1:** Pick one model and try the same prompt at different temperatures. What changes? Run each cell a few times to see the results.

Note: you'll need to change `X` in the sample code below to whichever one you chose.  Try to choose different models than your groupmates.

In [ ]:
# Low temperature - more predictable
ask_X("your prompt here", temperature=0.2)

In [ ]:
# medium temperature - more random
ask_X("your prompt here", temperature=1.2)

In [ ]:
# high temperature - very random
ask_X("your prompt here", temperature=1.9)

*Double-click this cell to paste examples and to write your thoughts:*



---

**Key takeaways:**

1. **The base model is just a text predictor** — it has no inherent "personality" or desire to be helpful.

2. **Helpfulness is trained, not inherent** — the instruct model learned to be helpful from thousands of examples.

3. **The same base can wear different masks** — we made it cheeky with just 150 examples of sarcastic/absurd student answers on test questions.

4. **Temperature reveals the "possibility space"** — higher temperature shows you more of what the model *could* say.

# Credits

This notebook contains materials created by Ethan C. Brown with assistance from Claude (see [conversation 1](https://claude.ai/share/58d28d97-ab83-48e0-b245-d533dc96d0d4), [conversation 2](https://claude.ai/share/108e259e-bb41-47d7-86ff-45b209f40fb1), and [conversation 3](https://claude.ai/share/670cb650-f7d7-45bc-afa9-a89916f13979)).